# Hex-grain burned area: how much burns, and is it predictable?

The companion to [`12_hex_ignition_baselines.ipynb`](12_hex_ignition_baselines.ipynb), which asks
*where do fires start*. This notebook asks the other half: **where do the acres land, and how much?**

They are different questions with different physics, and the distinction is the point:

> Fuel load probably does not decide **whether** a fire starts — ignition sources (lightning, roads,
> people) decide that. It very plausibly decides **how far one runs** once started.

That makes burned area the target where a fuel-density covariate has a mechanism to work through,
and it is the target the project's stakeholder actually cares about: a planner siting mitigation
wants to know where the *damage* lands, not only where starts occur.

**Why this needs perimeters and notebook 12 does not.** The mirror image of the W5 finding. FPA-FOD
stores a pinpoint ignition location but `FIRE_SIZE` describes an area:

| target | geometry | why |
| --- | --- | --- |
| ignition counts (nb 12) | raw points | the record stores ignition location correctly; perimeters would smear one start across ~26 hexes |
| burned acres (this nb) | MTBS perimeters | a fire larger than a hex (62,494 ac) provably cannot fit in the cell its ignition falls in |

Acres come from `data/hex_acres_res5.parquet` via [`../src/hex_burn.py`](../src/hex_burn.py), where
each fire's acreage is distributed across the hexes it actually covered with weights summing to 1.0.

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import hex_acres as ha
from config import ProjectConfig
from hex_panel import rank_score

warnings.filterwarnings("ignore")
cfg = ProjectConfig()
DATA = cfg.data
RNG = np.random.default_rng(0)

print(f"forward-chaining split: train < {cfg.test_start}, test >= {cfg.test_start}")
print(f"acres persistence window: k={ha.ACRES_K}")

## The target, and why its distribution dictates everything

Burned acres are not a normal regression target. Measured on Natural acres at res-5 over the full
record, the mass sits in a handful of cells.

In [ ]:
panel = ha.build_cached(DATA)

nz = panel[panel["acres_natural"] > 0]["acres_natural"]
print(f"nonzero natural-acre cells: {len(nz):,} of {len(panel):,} ({len(nz)/len(panel):.2%})\n")
print(nz.describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())

s = nz.sort_values(ascending=False)
print(f"\ntop  1% of burning cells hold {100 * s.head(int(len(s)*.01)).sum() / s.sum():.1f}% of natural acres")
print(f"top 10% of burning cells hold {100 * s.head(int(len(s)*.10)).sum() / s.sum():.1f}%")

**Two consequences drive every design choice below.**

1. **Model `log10(acres)`, not acres.** Five orders of magnitude separate the median burning cell
   (1 acre) from the maximum (606,945). On the raw scale a single megafire cell dominates any fit,
   and the model learns that cell rather than the phenomenon.

2. **The target contains two different questions**, and one model asked both will answer only the
   first, because 96% of rows are zeros:

   - **occurrence** — will this hex burn at all this season?
   - **magnitude** — given that it burns, how much?

   `ha.hurdle_frames()` returns them separately.

## Two baselines, and the bug that came from conflating them

Both are trailing means over a hex's own same-season history, via
[`../src/trailing.py`](../src/trailing.py). They are **not** interchangeable:

| baseline | averages over | answers |
| --- | --- | --- |
| `pers_log_*` | every prior season, zeros included | "how much does this hex burn per season on average" |
| `persburn_log_*` | prior **burning** seasons only | "when this hex burns, about how much" |

**The bug, kept here because it reversed a finding.** Scoring magnitude against `pers_log_*` gave
Natural a Spearman of **−0.052** — read at the time as "burn size is unpredictable from history, and
if anything history is mildly anti-informative."

That was an artifact. A hex that has never burned carries the `LOG_FLOOR` placeholder (−4, meaning
"no history"), which is not a prediction of 0.0001 acres — but it was being scored as one. **18% of
burning cells were in that state**, and comparing a placeholder against a real 600,000-acre burn
produced a nonsense "error" of 10⁸×.

Conditioning the baseline on burning seasons only, and dropping cells with no prior burn as
genuinely unscoreable, reverses the result entirely. The cell below shows both.

In [ ]:
test_mask = panel["season_year"] >= cfg.test_start
jja = panel["season_ord"] == 2

rows = []
for surface in ["natural", "human"]:
    burning = panel[jja & test_mask & (panel[f"acres_{surface}"] > 0)]

    # WRONG: all-season baseline, LOG_FLOOR placeholders scored as predictions.
    w = burning[burning[f"pers_log_{surface}"].notna()]
    rows.append({
        "surface": surface, "baseline": "pers_log (all seasons)", "n": len(w),
        "spearman": rank_score(w[f"log_{surface}"], w[f"pers_log_{surface}"]),
        "median_x_off": 10 ** np.median(np.abs(w[f"log_{surface}"] - w[f"pers_log_{surface}"])),
    })

    # RIGHT: burn-conditional baseline, no-history cells excluded.
    r = burning[burning[f"persburn_log_{surface}"].notna()]
    rows.append({
        "surface": surface, "baseline": "persburn_log (burning only)", "n": len(r),
        "spearman": rank_score(r[f"log_{surface}"], r[f"persburn_log_{surface}"]),
        "median_x_off": 10 ** np.median(np.abs(r[f"log_{surface}"] - r[f"persburn_log_{surface}"])),
    })

print("JJA magnitude, held-out years — the same data scored two ways\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nshare of burning cells whose all-season baseline is the LOG_FLOOR placeholder: "
      f"{(panel[jja & test_mask & (panel['acres_natural'] > 0)]['pers_log_natural'] <= -3.99).mean():.1%}")

**Finding — burn size *is* predictable from history, once the baseline asks the right question.**

Natural goes from −0.052 to **+0.37**, and the typical miss from ~370× to **6.3×**. The negative
result was measuring a placeholder competing with a real signal, not the absence of one.

This is worth stating plainly because the erroneous version was internally coherent: it had a
plausible magnitude, the right sign for "history doesn't help", and it agreed with a prior
expectation. Nothing about the number itself flagged it. What flagged it was that a 10⁸× error is
not physically possible.

## Is any of it better than chance?

The same shuffled-persistence control used in notebook 12, now applied to acres for the first time.
It holds the **exact set of predicted values** and destroys only the cell-to-cell mapping, so it
isolates *spatial* skill from the ability to emit plausible-looking magnitudes.

Both stages of the hurdle are scored: occurrence over all cells, magnitude over burning cells with
prior burn history.

In [ ]:
results = []
for surface in ["natural", "human"]:
    # --- occurrence: will this hex burn at all (all seasons) ---
    occ = panel[test_mask & panel[f"pers_log_{surface}"].notna()]
    y_occ = occ[f"burned_{surface}"].to_numpy(float)
    p_occ = occ[f"pers_log_{surface}"].to_numpy()
    results.append({
        "stage": "occurrence (does it burn)", "surface": surface, "n": len(occ),
        "floor": rank_score(y_occ, p_occ),
        "shuffled": ha.shuffled_null(y_occ, p_occ, rng=RNG),
    })

    # --- magnitude: how much given it burns (JJA) ---
    _, mag = ha.hurdle_frames(panel, surface, cfg=cfg, season_ord=2)
    mag = mag[mag["season_year"] >= cfg.test_start]
    y_mag = mag[f"log_{surface}"].to_numpy()
    p_mag = mag[f"persburn_log_{surface}"].to_numpy()
    results.append({
        "stage": "magnitude (how much)", "surface": surface, "n": len(mag),
        "floor": rank_score(y_mag, p_mag),
        "shuffled": ha.shuffled_null(y_mag, p_mag, rng=RNG),
    })

scores = pd.DataFrame(results)
print("Held-out season_year >= 2010. Spearman; shuffled = same values, wrong hexes.\n")
print(scores.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

**Finding — both stages carry real, spatial skill.**

Every shuffled control sits within ±0.008 of zero while every floor is between +0.34 and +0.51. As
in notebook 12, the skill lives in *which hex gets which number*.

Reading across the two notebooks, the picture is consistent: **a hex's own history is a strong
predictor of where fire happens and roughly how big it gets** — and it is the thing every external
covariate has so far failed to improve on.

## Where the baseline is weak — and why that is the interesting part

A median miss of 6.3× is decent for a quantity spanning five orders of magnitude. But W4
([`07_natural_location.ipynb`](07_natural_location.ipynb)) found persistence under-predicting **every
one** of the six largest held-out Natural cells by 1–1.7 orders of magnitude at region grain.

If that pattern holds at hex grain, then the baseline is adequate on typical cells and fails
specifically on the cells that carry the acres — which is exactly where a fuel-load covariate should
matter, and exactly where it would be most valuable to a planner.

In [ ]:
_, mag = ha.hurdle_frames(panel, "natural", cfg=cfg, season_ord=2)
mag = mag[mag["season_year"] >= cfg.test_start].copy()
mag["log_err"] = mag["log_natural"] - mag["persburn_log_natural"]

# Error by size decile: is the miss uniform, or concentrated in the big cells?
mag["decile"] = pd.qcut(mag["acres_natural"].rank(method="first"), 10, labels=range(1, 11))
by_size = mag.groupby("decile", observed=True).agg(
    median_acres=("acres_natural", "median"),
    median_log_err=("log_err", "median"),
    n=("log_err", "size"),
)
by_size["x_off"] = 10 ** by_size["median_log_err"].abs()
by_size["direction"] = np.where(by_size["median_log_err"] > 0, "UNDER-predicts", "over-predicts")

print("JJA natural magnitude error by burned-area decile (held-out years)\n")
print(by_size.to_string(float_format=lambda x: f"{x:.2f}"))

**Finding — the W4 pattern holds at hex grain, as a smooth gradient.** The error is not uniform.
The baseline over-predicts small cells, is nearly exact in the middle (decile 6, 1.07x), and
under-predicts the large ones — with the miss growing monotonically to **270x on the top decile**.

**In plain terms.** A patch's own history tells you it is fire country. It does not tell you when
that patch is going to have its worst year on record.

The useful analogy is rainfall. Past averages predict ordinary storms well and hurricanes not at
all — and the hurricane is the one you needed to prepare for.

**Why this is the finding that matters, and not a technical footnote.** Recall the concentration
measured at the top of this notebook: the top 1% of burning cells carry **55% of all natural acres**
and the top 10% carry **98%**. Lay that against the table above and the two facts combine into one
uncomfortable statement:

> The baseline is reliable on the cells that hold almost none of the acres, and unreliable on the
> cells that hold nearly all of them.

A median error of 6.3x is therefore a misleading summary of this model. The median cell burns
1 acre. The cells that decide whether a season is catastrophic are in decile 10, where the baseline
is off by more than two orders of magnitude — in the direction that matters, under-predicting.

**What that means for the planner.** It splits the decision cleanly:

| the question | does history answer it? |
| --- | --- |
| *Where do I site permanent works* — fuel breaks, thinning, defensible space | **Yes.** The map is stable; occurrence and typical magnitude are both predictable well above chance. |
| *Which places are about to have a catastrophic season* | **No.** This is precisely where the baseline fails, and it is the open question. |

That second row is the specific claim a fuel-density covariate has to beat — not the median cell,
but the top decile.

## Pending: does pre-season fuel density close the gap?

*(To be completed — the MODIS fetch in [`../src/hex_ndvi.py`](../src/hex_ndvi.py) is partially
built. This section runs the ablation once it finishes.)*

The ladder to run, mirroring notebook 12 so the two are directly comparable:

| rung | features |
| --- | --- |
| floor | `persburn_log_*` |
| + climate | pre-season PDSI / soil moisture / deficit / VPD |
| + NDVI (raw) | pre-season mean vegetation index |
| + NDVI (anomaly) | this hex against its own normal — the interannual part |

**The prior, stated before the result.** Three covariates have now failed to beat persistence on
*ignition counts*, and the reason measured in notebook 12 was that their signal is **cross-sectional**
— they identify dry places, not dry years. NDVI shares that structure: forests are green every year.

But two things differ here and justify running it rather than assuming the answer:

1. **The mechanism is more plausible.** Fuel load should govern how far a fire runs, not whether it
   starts.
2. **The baseline has a measured, specific weakness** — the top-decile under-prediction above —
   whereas on ignition counts persistence was strong everywhere and there was no gap to fill.

If NDVI fails here too, that is a substantive finding rather than a fourth disappointment: it would
say burned area at this grain is governed by ignition location and terrain, with pre-season fuel
state adding nothing a hex's own history does not already carry.

## Summary

**For an analyst.**

1. **Burned area is predictable above chance, and the skill is spatial.** Occurrence floors are
   +0.34 (natural) and +0.51 (human); magnitude floors are +0.37 and +0.45. Every shuffled control —
   same values, wrong hexes — sits within ±0.008 of zero.
2. **The two stages must be modelled separately.** 96% of hex-seasons have no fire; a single model
   asked both "does it burn" and "how much" answers only the first.
3. **The baseline must be conditioned on burning seasons.** Scoring magnitude against an all-season
   mean produced −0.052 and the conclusion "burn size is unpredictable" — an artifact of `LOG_FLOOR`
   placeholders being scored as predictions for 18% of cells. Corrected, the same data gives +0.37.
4. **The gap is in the tail.** Error grows monotonically with burned area, from 9x over-prediction
   in decile 1 to 270x under-prediction in decile 10, reproducing the W4 megafire finding at hex
   grain.

**In plain language.**

For each 10-km patch we asked two questions — *will it burn this summer*, and *if it does, how
much* — and answered both using nothing but that patch's own track record. No weather, no
satellite data.

Both are answerable better than chance. Patches that burned before tend to burn again, and patches
that burned big tend to burn big again. Shuffling the predictions onto the wrong patches destroys
the skill entirely, which is how we know it is real.

But accuracy collapses with fire size. The prediction over-guesses on tiny fires, is almost exactly
right on typical ones, and badly under-guesses on the big ones — off by 270x on the largest. And the
largest are where everything is at stake: **the top 1% of burning patches account for 55% of all
acres burned, the top 10% for 98%.**

So the model is dependable on the fires that do not matter much, and unreliable on the ones that do.
A place's history tells you it is fire country; it does not tell you when that place is going to
have its worst year.

**A caveat worth carrying with this result.** An earlier version of this analysis concluded that burn
size was *unpredictable* from history. That was a bug — patches with no fire record were being
scored as though we had predicted "0.0001 acres" rather than "we do not know." The same data, scored
correctly, flipped from no signal to real signal.

It is worth stating because the wrong answer looked entirely reasonable: plausible magnitude, the
expected sign, consistent with a prior belief. Nothing about the number itself gave it away. What
gave it away was noticing that one implied error worked out to 100-million-fold, which is not
physically possible.

**Open:** the fuel-density ablation above, and whether the top-decile under-prediction is closeable
by any pre-season covariate.